In [ ]:
%pip install -q langchain langchain-openai langchain-chroma langchain-community chromadb python-dotenv

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv()

DATA_PATH = "private_data"

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

In [ ]:
def enrich_chunk(doc):
    prompt = f"""
    Create a short retrieval-friendly summary of this text.

    Text:
    {doc.page_content}

    Return only the summary.
    """

    summary = llm.invoke(prompt).content

    doc.metadata["summary"] = summary
    return doc


chunks = [enrich_chunk(chunk) for chunk in chunks]

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_private_rag",
    persist_directory="./chroma_db"
)

print("Vector database created.")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 8}
)

question = "How does authentication work in my project?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:500])

In [ ]:
def rerank(question, documents):

    numbered = "\n\n".join(
        f"[{i}] {doc.page_content}"
        for i, doc in enumerate(documents)
    )

    prompt = f"""
    Rank these documents from most relevant to least relevant
    for answering the question.

    Question:
    {question}

    Documents:
    {numbered}

    Return only the document numbers separated by commas.
    """

    response = llm.invoke(prompt).content

    order = [
        int(x.strip())
        for x in response.split(",")
    ]

    return [documents[i] for i in order]


reranked_docs = rerank(question, retrieved_docs)

In [ ]:
def answer_question(question):

    docs = retriever.invoke(question)
    docs = rerank(question, docs)

    context = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n"
        f"Summary: {doc.metadata.get('summary', '')}\n"
        f"{doc.page_content}"
        for doc in docs
    )

    prompt = f"""
    Answer the question using only the provided context.

    If the answer cannot be found in the context, say:
    "I don't have enough information in the knowledge base."

    Question:
    {question}

    Context:
    {context}
    """

    response = llm.invoke(prompt)

    return response.content, docs

In [ ]:
answer, sources = answer_question(
    "How does authentication work in my project?"
)

print(answer)

print("\nSources:")
for doc in sources:
    print(doc.metadata.get("source"))